In [1]:
import pickle
import pandas as pd
import numpy as np
import gc

## RF

In [2]:
filepath_base_rf = "model/debit/base_rf_ros_96_feats_20250623.pkl"
with open(filepath_base_rf, "rb") as f:
    base_rf = pickle.load(f)

## Data

In [9]:
df_final = pd.read_parquet("data/ready_to_train/df_debit_final_raw_20250623.parquet")

In [10]:
# setup pipeline
from src.model_pipeline import ModelPipeline

pipeline_rf = ModelPipeline(model_type="random_forest", random_state=42)

split_date = "2025-05-01"
df_splits = pipeline_rf.split_data_by_date(
    df=df_final, date_column="Transaction Datetime", split_date=split_date
)

Initializing ModelPipeline with model type: random_forest
Retrieving model instance for type: random_forest
Splitting data by date (pre-preprocessing)...
Train samples: 647760, Test samples: 66974
Data splitting complete.


In [11]:
# prepare OOS data
target_col = "Confirmed"
X, y = pipeline_rf.prepare_data(
    df=df_splits["df_train"],
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    is_apply_one_hot=True,
    is_apply_log=False,
    is_impute_median=False,
    is_training=True,
)

Preparing data (is_training=True)...
Starting core numeric and boolean preprocessing...
Core numeric and boolean preprocessing complete.
Fitting CategoryManager...
  Fitted categories for column 'AccountStatus': ['1', '2', '3', '6', '7', '8', '__missing__']
  Fitted categories for column 'Currency': ['IDR', '__missing__']
  Fitted categories for column 'CustomerSex': ['F', 'M', '__missing__']
  Fitted categories for column 'HIghRiskCustomer': ['0', 'N', '__missing__']
  Fitted categories for column 'POSMode': ['0', '1', '2', '5', '7', '9', '__missing__']
  Fitted categories for column 'TransactionType': ['0', '1', '23', '31', '4', '40', '50', '51', 'BI', 'CA', 'DC', 'FT', 'IN', 'PA', '__missing__']
  Fitted categories for column 'CardProductGrouped': ['CARD_PRODUCT_1', 'CARD_PRODUCT_2', 'CARD_PRODUCT_3', 'CARD_PRODUCT_4', 'CARD_PRODUCT_5', 'CARD_PRODUCT_6', 'CARD_PRODUCT_7', 'CARD_PRODUCT_OTHER', '__missing__']
  Fitted categories for column 'CountryCodeGroup': ['AUS', 'IDN', 'MYS', 'O

In [12]:
X_oos, y_oos = pipeline_rf.prepare_data(
    df=df_splits["df_test"],
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    is_apply_one_hot=True,
    is_apply_log=False,
    is_impute_median=False,
    is_training=False,
)

Preparing data (is_training=False)...
Starting core numeric and boolean preprocessing...
Core numeric and boolean preprocessing complete.
Transforming data using CategoryManager...
CategoryManager transformation complete.
Applying One-Hot Encoders for column: Index(['AccountStatus', 'Currency', 'CustomerSex', 'HIghRiskCustomer',
       'POSMode', 'TransactionType', 'CardProductGrouped', 'CountryCodeGroup'],
      dtype='object')...
Identified numerical columns for imputation: ['Avg_Amt_L15M', 'Avg_Amt_L30D', 'Avg_Amt_to_CountryCode_L15M', 'Avg_Amt_to_CountryCode_L30D', 'Avg_Amt_to_MCC_L15M', 'Avg_Amt_to_MCC_L30D', 'CntUnique_CardNo_by_MCC_L15M', 'CntUnique_CardNo_by_MCC_L30D', 'Max_Amt_L15M', 'Max_Amt_L30D', 'Max_Amt_to_CountryCode_L15M', 'Max_Amt_to_CountryCode_L30D', 'Max_Amt_to_MCC_L15M', 'Max_Amt_to_MCC_L30D', 'Sum_Amt_L30D', 'Sum_Amt_to_CountryCode_L15M', 'Sum_Amt_to_CountryCode_L30D', 'Sum_Amt_to_MCC_L15M', 'Sum_Amt_to_MCC_L30D', 'TxnCount_L15M', 'TxnCount_L30D', 'TxnCount_to_Cou

# Binning Proba Analysis

In [13]:
from src.utils import plot_confusion_matrix, generate_pred_df, compute_bin_aggregates

In [14]:
df_pred = generate_pred_df(
    X=X_oos,
    y=y_oos,
    clf=base_rf,
    amount_col="Transaction Amount",
    bin_on="pbad",
    increment_bin_rate=0.001,
    bins=None,
)

In [15]:
df_agg = compute_bin_aggregates(df_pred)

C:\Users\Administrator\BDI-predator\ml-fraud\src\utils.py:115: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  aggregated_df = pred_df.groupby("bin").agg(agg_cols)


In [16]:
df_agg.to_csv("data/result/debit_rf_bin_proba_analysis_20250623.csv")